# Tshivenda Misinformation Classifier - Colab

Bootstrap notebook: run top to bottom on a **fresh Colab GPU runtime** to fine-tune
and cross-validate the Tshivenda misinformation classifier much faster than the
local M4 run (minutes instead of ~an hour).

**Context**: the Mukwevho et al. (2024) Tshivenda misinformation dataset is not
accessible to this team (never publicly released). This trains on a synthetic
proxy built from the Vukuzenzele Tshivenda text instead - see
`notes/tshivenda-classifier-proxy.md` in the repo for the full rationale and
honest limitations before citing these numbers anywhere.

Two models, both evaluated with grouped 5-fold cross-validation (179 source
articles is too small for a single train/test split to mean much):
- `Davlan/afro-xlmr-base` - primary, per the proposal.
- `xlm-roberta-base` - comparison baseline.

**Before you start:** Runtime -> Change runtime type -> GPU (T4 is plenty for
these model sizes / dataset size - this is a much lighter job than ASR training).

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2. Clone the repo + install dependencies

The proxy dataset (`dataset/vukuzenzele/misinfo_proxy_ven.csv`) is small and already committed - no download/regeneration needed, unlike the ASR audio.

In [ ]:
!git clone -b dev https://github.com/Khotso-Bore/MultilingualASR.git
%cd MultilingualASR
!pip install -q -r requirements.txt

## 3. (Optional) rebuild the proxy dataset

Only needed if you changed src/classification/build_misinfo_proxy_ven.py - the committed CSV is used as-is otherwise.

In [ ]:
# !python src/classification/build_misinfo_proxy_ven.py

## 4. Train + cross-validate: AfroXLM-RoBERTa (primary)

In [ ]:
!python src/classification/train_classifier_ven.py --model Davlan/afro-xlmr-base --folds 5 --epochs 15 --learning-rate 1e-3 --freeze-base \
    | tee /content/afroxlmr_results.log

## 5. Train + cross-validate: XLM-RoBERTa (comparison)

In [ ]:
!python src/classification/train_classifier_ven.py --model xlm-roberta-base --folds 5 --epochs 15 --learning-rate 1e-3 --freeze-base \
    | tee /content/xlmr_results.log

## 6. Compare

In [ ]:
import re

def parse(path):
    text = open(path).read()
    acc = re.search(r"accuracy: ([\d.]+) \+/- ([\d.]+)", text)
    f1 = re.search(r"macro F1: ([\d.]+) \+/- ([\d.]+)", text)
    return float(acc[1]), float(acc[2]), float(f1[1]), float(f1[2])

a_acc, a_acc_sd, a_f1, a_f1_sd = parse("/content/afroxlmr_results.log")
x_acc, x_acc_sd, x_f1, x_f1_sd = parse("/content/xlmr_results.log")

print(f"{'model':<20} {'accuracy':>16} {'macro F1':>16}")
print(f"{'afro-xlmr-base':<20} {a_acc:.3f} +/- {a_acc_sd:.3f}   {a_f1:.3f} +/- {a_f1_sd:.3f}")
print(f"{'xlm-roberta-base':<20} {x_acc:.3f} +/- {x_acc_sd:.3f}   {x_f1:.3f} +/- {x_f1_sd:.3f}")

## 7. Save results back to Drive

So the logs survive after the Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
DRIVE_DIR = "/content/drive/MyDrive/multilingualasr/classifier-results"
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy("/content/afroxlmr_results.log", DRIVE_DIR)
shutil.copy("/content/xlmr_results.log", DRIVE_DIR)
print(f"saved -> {DRIVE_DIR}")